# Backtest Notebook

This notebook generates (or reuses) `market_data.csv`, runs the backtest, and displays metrics, periodic returns, and an equity curve plot.


In [ ]:
import os
from data_generator import generate_market_csv

CSV_PATH = "market_data.csv"

if not os.path.exists(CSV_PATH):
    generate_market_csv(
        symbol="AAPL",
        start_price=150.0,
        filename=CSV_PATH,
        num_ticks=500,
        volatility=0.02,
        interval=0.0,
    )
    print("Generated", CSV_PATH)
else:
    print("Found existing", CSV_PATH)


In [ ]:
from data_loader import load_market_data
from engine import BacktestEngine
from reporting import periodic_returns, total_return, sharpe_ratio, max_drawdown
from strategies import MovingAverageCrossoverStrategy, MomentumStrategy

ticks = load_market_data(CSV_PATH)
symbol = ticks[0].symbol

strategies = [
    MovingAverageCrossoverStrategy(symbol=symbol, short_window=5, long_window=20, order_size=10),
    MomentumStrategy(symbol=symbol, lookback=10, threshold=0.01, order_size=10),
]

# Set fail_rate=0 for deterministic execution (no simulated failures)
engine = BacktestEngine(strategies=strategies, initial_cash=10_000.0, fail_rate=0.0)
result = engine.run(ticks)

equity_curve = result.equity_curve
initial_eq = equity_curve[0][1]
final_eq = equity_curve[-1][1]
rets = periodic_returns(equity_curve)

metrics = {
    "Total Return": total_return(initial_eq, final_eq),
    "Sharpe Ratio": sharpe_ratio(rets),
    "Max Drawdown": max_drawdown(equity_curve),
}

metrics


In [ ]:
# Metrics table
print("Metric".ljust(15), "Value")
print("-" * 30)
print("Total Return".ljust(15), f"{metrics['Total Return']:.2%}")
print("Sharpe Ratio".ljust(15), f"{metrics['Sharpe Ratio']:.4f}")
print("Max Drawdown".ljust(15), f"{metrics['Max Drawdown']:.2%}")
print()
print(f"Initial equity: {initial_eq:.2f}")
print(f"Final equity:   {final_eq:.2f}")


In [ ]:
# Periodic return series
print("Periodic returns count:", len(rets))

if rets:
    mean_r = sum(rets) / len(rets)
    var_r = sum((r - mean_r) ** 2 for r in rets) / (len(rets) - 1) if len(rets) > 1 else 0.0
    std_r = var_r ** 0.5
    print(f"Mean: {mean_r:.8f}")
    print(f"Std:  {std_r:.8f}")
    print(f"Min:  {min(rets):.8f}")
    print(f"Max:  {max(rets):.8f}")

print()
print("First 20 returns (index, return):")
for i, r in enumerate(rets[:20], start=1):
    print(i, f"{r:.8f}")


In [ ]:
# Equity curve plot
vals = [eq for _, eq in equity_curve]
times = [t.timestamp for t, _ in equity_curve]

try:
    import matplotlib.pyplot as plt
    plt.figure()
    plt.plot(times, vals)
    plt.title("Equity Curve")
    plt.xlabel("Time")
    plt.ylabel("Equity")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("Plot skipped:", e)
    # Simple ASCII fallback
    chars = " .:-=+*#%@"
    width = 60
    if len(vals) > width:
        step = (len(vals) - 1) / (width - 1)
        sample = [vals[int(round(i * step))] for i in range(width)]
    else:
        sample = vals
    lo, hi = min(sample), max(sample)
    if hi == lo:
        print("." * len(sample))
    else:
        out = []
        for v in sample:
            t = (v - lo) / (hi - lo)
            idx = int(round(t * (len(chars) - 1)))
            out.append(chars[idx])
        print("".join(out))


## Narrative

- **Total return** is the overall change in equity from start to finish.
- **Periodic returns** are tick-to-tick equity changes (used to compute Sharpe).
- **Sharpe ratio** here is based on tick returns (not annualized). Use it mainly to compare runs on the same data/settings.
- **Max drawdown** is the worst peak-to-trough loss in the equity curve.
